### Data Ingestion Pipeline

In [31]:
# Document Data Structure
from langchain_core.documents import Document

doc = Document(
    page_content="Akshay helps developers build LLM applications, with through a standard interface for all LLMs, a framework for building applications that use LLMs, and integrations with other sources of data.",
    metadata={
        "source": "tutorial.pdf", "page": 1, "Author": "Keerti Mahesh", " Date": "2023-06-01"}
)

# Keeping down metadata is used when applying filters


print(doc.page_content)
print(doc.metadata)

Akshay helps developers build LLM applications, with through a standard interface for all LLMs, a framework for building applications that use LLMs, and integrations with other sources of data.
{'source': 'tutorial.pdf', 'page': 1, 'Author': 'Keerti Mahesh', ' Date': '2023-06-01'}


In [32]:
#Creating a Simple Document
import os
os.makedirs("../data/text_files", exist_ok=True)
os.makedirs("../data/PDF_files", exist_ok=True)

In [33]:
#Pushing Down the content in the doc file

sample_text = {"../data/text_files/python_intro.txt": """LangChain helps developers build LLM applications, with through a standard interface for all LLMs,
a framework for building applications that use LLMs, and integrations with other sources of data."
"""
}


for file_path, content in sample_text.items():
    with open(file_path, "w",  encoding = "utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [34]:
import sys
print(sys.executable)

c:\Users\12116\OneDrive\Desktop\RAG\.venv\Scripts\python.exe


In [35]:
# Reading the content of the file using TextLoader

#from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding = "utf-8")  #This gives the object of the text Loader

document = loader.load()  #This will give the content of the file in the form of a list of documents
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='LangChain helps developers build LLM applications, with through a standard interface for all LLMs,\na framework for building applications that use LLMs, and integrations with other sources of data."\n')]


In [36]:
# Reading all the important listing in the directory - Directory Loader

from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader("../data/text_files", glob = "**/*.txt", loader_cls = TextLoader, loader_kwargs = {'encoding': 'utf-8'}, show_progress = False)  #This will give the object of the Directory Loader

documents = dir_loader.load()  #This will give the content of the file in the form of a list of documents
documents

[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='LangChain helps developers build LLM applications, with through a standard interface for all LLMs,\na framework for building applications that use LLMs, and integrations with other sources of data."\n'),
 Document(metadata={'source': '..\\data\\text_files\\Salesforce.txt'}, page_content="2) What had attracted to apply for this role?\n\nI am very much interested in applying for the position of a Technical Support Engineer because I find myself intresting when i solve up technical  problems that ideally satisfies the customers to have the best possible experience.I find staisfaction in understanding the root cause of the issue by brainstorming a lot to think differently and understand a problem before defining or figuring out a known solution that you think it will fit in well. I my previous role as a Forward Deployed Engineer, I often handled late-night support calls and collaborated with them to reso

In [37]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

#Loading the text files from the directory

dir_loader = DirectoryLoader("../data/PDF_files", glob="**/*.pdf", loader_cls=PyMuPDFLoader, show_progress=False)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'file_path': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='han lonnai. \npm \n&hk tintbrgnd in \nl. \nutsuing \nZudihh \ndeonoe nd bginatriy \nChorrai. haa cogolataa ag Soheeting hen on bano rne, \nChonnai. hae semeus aohaieand in he kaldgpnk \nond alo pravntea Lanibnad \nTNOIA pma. had al prtizgotaa in meny kpa \nakenal ariad'),
 Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Inte

In [38]:
type(pdf_documents[0])
type(pdf_documents)

list

### Splitting the docs into different specific Chunks - Data Ingestion to vector DB pipeline

In [39]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [40]:
#Reading all the PDF's inside the directory

def process_all_pdfs(pdf_directory):
    """Processes all PDF files in the specified directory and returns a list of documents."""

    all_documents = []
    pdf_directory_path = Path(pdf_directory)

    # Finding all the PDF and subpdfs inside the directory
    pdf_files = list(pdf_directory_path.glob("**/*.pdf"))


    print(f" Found {len(pdf_files)} PDF files to process in the directory: {pdf_directory_path}")

    for pdf_file in pdf_files:
        print(f"\n Processing: {pdf_file.name}")

        try:

            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #Source information for Meta Data

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error processing {pdf_file.name}: {e}")


    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


#Processing all PDF's in the document

all_pdf_documents = process_all_pdfs("../data")


 Found 2 PDF files to process in the directory: ..\data

 Processing: Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf
Loaded 3 pages

 Processing: Striver A2Z Full Notes - all approaches included, Prepared by Vinay kajla IG- Vinay.Kajla_.pdf
Loaded 1951 pages

Total documents loaded: 1954


In [41]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'file_type': 'pdf'}, page_content='han lonnai. pm \n&hk tintbrgnd in l. \nutsuing Zudihh deonoe nd bginatriy \nChorrai. haa cogolataa ag Soheeting hen on bano rne, \nChonnai. hae semeus \naohaieand in he kaldgpnk \nond alo pravntea \nLanibnad \nTNOIA pma. had al prtizgotaa in meny kpa \nakenal ariad'),
 Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2', 'source_file': 

In [42]:
### Chunking the docs - Recursive Character Text Splitter

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Splitting the document into recursive Character Text Splitter"""

    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap, length_function = len, separators = ["\n\n", "\n",",", " ", ""])

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks") 



    #Example of the first chunk and its metadata
    if split_docs:
        print(f"Example Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs
    


In [43]:
chunks = split_documents(all_pdf_documents)
chunks

Split 1954 documents into 2951 chunks
Example Chunk:
Content: han lonnai. pm 
&hk tintbrgnd in l. 
utsuing Zudihh deonoe nd bginatriy 
Chorrai. haa cogolataa ag Soheeting hen on bano rne, 
Chonnai. hae semeus 
aohaieand in he kaldgpnk 
ond alo pravntea 
Lanibnad...
Metadata: {'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'file_type': 'pdf'}, page_content='han lonnai. pm \n&hk tintbrgnd in l. \nutsuing Zudihh deonoe nd bginatriy \nChorrai. haa cogolataa ag Soheeting hen on bano rne, \nChonnai. hae semeus \naohaieand in he kaldgpnk \nond alo pravntea \nLanibnad \nTNOIA pma. had al prtizgotaa in meny kpa \nakenal ariad'),
 Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2', 'source_file': 

### Embeddings and Vector Store DB - text into vectors

In [44]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #used for inserting the unique id of the document
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


print("All Imports are successful!")

All Imports are successful!


In [45]:
#Contains the Model Information 

class EmbeddingManager:
    """Handles document creation and embedding generation using Sentence Transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialising the Embedding Manager
        
        Args:
            model_name (str): Hugging Face model name for Sentence Embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model() # _is used to indicate that this is a protected function and is accessed only within the class or its subclasses.


    def _load_model(self):

        """Load the Sentence Transformer model"""

        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully!. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}") #By Default, the embedding dimension for the model is 384

        except Exception as e:
            print(f"Error Loading the Model: {e}")
            raise e


    def generate_embeddings(self, documents: List[str]) -> np.ndarray:
        """Generate Embeddings for a list of documents
        
        Args:
            documents (List[str]): List of texts to get embedded
            
        Returns:
            Numpy array of Embeddings with shape (len(documents), embedding_dimension)"""

        if not self.model:
            raise ValueError("Model is not Loaded.")

        print(f"Generating Embeddings for {len(documents)} documents....")
        embeddings = self.model.encode(documents, show_progress_bar = True, convert_to_numpy = True)
        print(f"Embeddings Generated Successfully!. Shape: {embeddings.shape}")
        return embeddings




# Intialising the Embedding Manager

embedding_manager = EmbeddingManager()
embedding_manager



Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4604.36it/s]


Model Loaded Successfully!. Embedding Dimension: 384


C:\Users\12116\AppData\Local\Temp\ipykernel_30004\4027537958.py:25: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Successfully!. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}") #By Default, the embedding dimension for the model is 384


Vector Store

In [55]:
class VectorStore:
    
    """Manages Document Embeddings in ChromaDB Vector Store"""


    def __init__(self, collection_name = "pdf_documents", persistent_directory  = "../data/vector_store"):
        """Initialising the Vector Store
        
        Args:
            Collection Name : Name of the ChromaDB Collection to the store the embeddings
            Persistent Directory: Path to the directory where the embeddings will be stored
        
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._intialise_store()  #Protected Function in the Vector Class


    def _intialise_store(self):
        """Intialising the ChromaDB Client and collection"""

        try:

            #Creating Persistent ChromaDB Client
            os.makedirs(self.persistent_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persistent_directory)


            #Get Or Create the Collection

            self.collection  = self.client.get_or_create_collection(name = self.collection_name, metadata = {"description" : "Collection of PDF Document Embeddings for the RAG Model", "embedding_model" : embedding_manager.model_name, "embedding_dimension" : embedding_manager.model.get_sentence_embedding_dimension(), "hnsw:space" : "cosine"})


            print(f"Vector Store Intialised Successfully!. Collection: {self.collection_name}")
            print(f"Existing Documents in the Collection: {self.collection.count()}")


        except Exception as e:
            print(f"Error intialiasing vector store: {e}")
            raise e

    # Adding the documents to the vector store DB


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        """Adding Documents to the ChromaDB collection with their Embeddings
        
        Args:
        document: List of Langchain Documents
        Embddings: Corresponding Embeddings for the Documents
        
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of Documents must match number of Embeddings")


        print(f"Adding {len(documents)} documents to the vector store...")


        #Preparing up the data for ChromaDB

        uuids = []
        meta_datas = []
        document_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            #Generating the UUID
            doc_id = f"doc{uuid.uuid4().hex[:8]}_{i}"
            uuids.append(doc_id)


            #Preparing the metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            meta_datas.append(metadata)


            #Document Content
            document_text.append(doc.page_content)

            #Embedding
            embedding_list.append(embedding.tolist())



        #Add to Collections

        try:
            self.collection.add(ids = uuids, embeddings = embedding_list, documents = document_text, metadatas = meta_datas)
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total Documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store as {e}")
            raise e


vectorstore = VectorStore()
print(vectorstore)
#vectorstore
    

Vector Store Intialised Successfully!. Collection: pdf_documents
Existing Documents in the Collection: 11804


C:\Users\12116\AppData\Local\Temp\ipykernel_30004\1213572753.py:34: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.collection  = self.client.get_or_create_collection(name = self.collection_name, metadata = {"description" : "Collection of PDF Document Embeddings for the RAG Model", "embedding_model" : embedding_manager.model_name, "embedding_dimension" : embedding_manager.model.get_sentence_embedding_dimension(), "hnsw:space" : "cosine"})


In [56]:
#Specific Group of chunks
chunks

[Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'file_type': 'pdf'}, page_content='han lonnai. pm \n&hk tintbrgnd in l. \nutsuing Zudihh deonoe nd bginatriy \nChorrai. haa cogolataa ag Soheeting hen on bano rne, \nChonnai. hae semeus \naohaieand in he kaldgpnk \nond alo pravntea \nLanibnad \nTNOIA pma. had al prtizgotaa in meny kpa \nakenal ariad'),
 Document(metadata={'producer': 'Adobe Scan for Android 22.03.07-regular', 'creator': 'Adobe Scan for Android 22.03.07-regular', 'creationdate': '', 'source': '..\\data\\PDF_files\\Interview questions _21BCE5324 - Gandlapalli Venkat Sai Akshay.pdf', 'total_pages': 3, 'page': 1, 'page_label': '2', 'source_file': 

In [57]:
#Extracting all the text from the specific chunk and generate an embedding text -> Embeddings


texts = [doc.page_content for doc in chunks]


#Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)


#Storing it up in the vector DB

vectorstore.add_documents(chunks,embeddings) #upper function is used to add the documents to the vector store

Generating Embeddings for 2951 documents....


Batches: 100%|██████████| 93/93 [00:42<00:00,  2.20it/s]


Embeddings Generated Successfully!. Shape: (2951, 384)
Adding 2951 documents to the vector store...
Sucessfully added 2951 documents to vector store
Total Documents in collection: 14755


## Retrieval Pipeline from the Vector Store

In [ ]:
class RAGRetriever:
    """Handles Query based Retrieval from the RAG store (basically from the RAG DB)"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Intialise the Retriever


        Args:
        
            VectorStore: Vector Store containing the document embeddings
            embedding_manager: Manager for generating query embeddings
        
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query: str, top_k: int = 5, score_threshold : float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve Relavent documents from the User Query
        
        Args
        - query: Search Query
        - top _k: Number of top results to return
        -score_threshold: Minimum similarity score threshold

        Returns: List of dictionaries containing retrieved documenets and metadata

        """

        print(f"Retrieving Documents from the Query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")


        #Generating the query Embedding

        #User Query is generated into an Embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #Search in Vector Store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #Processing up the results

            retrieved_docs= []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]


                #Creating a function for the similarity score match
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids,documents, metadatas, distances)):
                    # Converting the distance to the similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1- distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'distance' : distance,
                            'similarity_score' : similarity_score,
                            'rank' : i+1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering): ")

            
            else:
                print("No Documents Found")

            return retrieved_docs

        except Exception as e: 
            print(f"Error during retrieval : {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)
            

In [59]:
rag_retriever

In [60]:
rag_retriever.retrieve("Who is the priminister of India")

Retrieving Documents from the Query: 'Who is the priminister of India'
Top K: 5, Score Threshold: 0.0
Generating Embeddings for 1 documents....


Batches: 100%|██████████| 1/1 [00:00<00:00, 60.61it/s]

Embeddings Generated Successfully!. Shape: (1, 384)
Retrieved 0 documents (after filtering): 


[]

Integrating Vector FB Context Pipeline with LLM Output

In [73]:
# Simple RAG Pipeline with Google LLM

from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

# Intialising the Google LLM API Key

google_llm_api_key = os.getenv("Google_Api_Key")

llm = ChatGoogleGenerativeAI(google_api_key = google_llm_api_key, model = "gemini-3.8-flash", temperature = 0.1, max_tokens = 1024 )


# 2. Creating a RAG Function: retrieve_content + llm response

def rag_simple(query, retriever, llm, top_k: int = 3):
    #Retrieve the Context

    results = retriever.retrieve(query, top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    #if not context:
        #return "No Relevant Context found to answer the Question"


    #Generate the answer using the Google LLM

    prompt = f"""Use the following context to answer the question concisely.
    
    Context: {context}
    Question: {query}

    Answer:
    
    """

    response = llm.invoke(prompt)
    answer = "".join(
        block["text"]
        for block in response.content
        if isinstance(block, dict) and block.get("type") == "text"
    )

    return answer

In [74]:
answer = rag_simple("What is the attention Mechanism?", rag_retriever, llm)
print(answer)

Retrieving Documents from the Query: 'What is the attention Mechanism?'
Top K: 3, Score Threshold: 0.0
Generating Embeddings for 1 documents....


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.08it/s]

Embeddings Generated Successfully!. Shape: (1, 384)
Retrieved 0 documents (after filtering): 


The provided context does not contain any information to answer the question. 

Generally, the **attention mechanism** is a deep learning technique that allows a model to dynamically focus on the most relevant parts of an input sequence when making predictions or generating an output.


Enhanced RAG PIpeline Feature

In [63]:
# ---Enhanved RAG Pipeline Feature---

def rag_adavanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False):
    """RAG Pipeline with Extra Features:
    
    - Returns answers, sources of Truth, Confidence Score, Optionally the full provided context
    
    """

    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)
    if not results:
        return {'answer' : 'No relevant Context found','sources' : [], 'confidence' : 0.0, 'context' : ''}

    #Preparing the context and the truth of sources

    context = "\n\n" .join([doc['content'] for doc in results])

    sources = [{
        'source ': doc['metadata'].get('source_file', doc['metadata'].get('source' , 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:120] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])


    #Generate Answer

    prompt = f"""Use the following context to answer the Questions Concisely. \nContext:\n{context} \n\n Question: {query} \n\n Answer."""
    response = llm.invoke([prompt.format(context = context, query = query)])


    output = {
        'answer' : response.content,
        'sources' :sources,
        'confidence' : confidence
    }

    if return_context:
        output['context'] = context
    return output



#Example Usecase

result = rag_adavanced("Who is Sai Akshay, and what does he do", rag_retriever, llm, top_k = 3, min_score = 0.1, return_context = True)
print("Answer:" , result['answer'])
print("Sources:" , result['sources'])
print("Confidence: ", result['confidence'])
print("Context Preview: " , result['context'][:250])



Retrieving Documents from the Query: 'Who is Sai Akshay, and what does he do'
Top K: 3, Score Threshold: 0.1
Generating Embeddings for 1 documents....


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.17it/s]

Embeddings Generated Successfully!. Shape: (1, 384)
Retrieved 0 documents (after filtering): 
Answer: No relevant Context found
Sources: []
Confidence:  0.0
Context Preview:  


In [64]:

# Really Advanced RAG Pipeline: Streaming, Citations, History, Summarisations ----

from typing import List, Dict, Tuple, Any
import time

class AdvanvedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = [] #Storing up the Query History

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False):

        #Retrieving up th relevenat document

        results = self.retriever.retrieve(question, top_k = top_k, score_threshold = min_score)

        if not results:
            answer = "No Relevant Context Found."
            sources = []
            context = ""

        else:
            context = "\n\n".join([doc['content'] for doc in results])

            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:200] + '...'
            } for doc in results]

            #Streaming answer Stimulation

            prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {question}"""

            if stream:
                print("Streaming Answer: ")
                answer_parts = []

                for chunk in self.llm.stream(prompt):
                    content = chunk.content

                    if isinstance(content, str) and content:
                        print(content, end = '', flush = True)
                        answer_parts.append(content)

                print()
                answer = "".join(answer_parts)

            else:
                response = self.llm.invoke(prompt)
                answer = response.content

        # Adding citations to the answer

        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations)
            if citations else answer
        )

        #Optional Summary Answer

        summary = None

        if summarize and answer != "No Relevant Context Found.":
            summary_prompt = f"Summarise the following answer in 2 sentences:\n{answer}"
            summary_response = self.llm.invoke(summary_prompt)
            summary = summary_response.content

        # Store Query history

        history_entry = {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary
        }

        self.history.append(history_entry)

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'context': context,
            'history': self.history
        }


# Example usage:

adv_rag = AdvanvedRAGPipeline(rag_retriever, llm)

result = adv_rag.query(
    "What is attention all the you need",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


Retrieving Documents from the Query: 'What is attention all the you need'
Top K: 3, Score Threshold: 0.1
Generating Embeddings for 1 documents....


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.66it/s]

Embeddings Generated Successfully!. Shape: (1, 384)
Retrieved 0 documents (after filtering): 

Final Answer: No Relevant Context Found.
Summary: None
History: {'question': 'What is attention all the you need', 'answer': 'No Relevant Context Found.', 'sources': [], 'summary': None}
